# 文本排序与困难负样本：从 pair schema 到可观测 reranker

本 notebook 实现一个离线可运行的学习排序最小系统：

query/doc/judgment 合同 → group/time split → 手写 TF-IDF baseline → random/hard negative mining → pointwise/pairwise 线性 ranker → MRR/nDCG/Recall → 校准与阈值 → rerank budget/batch/trace/drift

它用小型受控数据解释工程边界，不宣称达到现代 cross-encoder 的排序质量。

## 1. 外部合同与业务目标

检索阶段返回 request_id、query_id、tenant_id、candidate_doc_ids、retriever/index_version；排序阶段只能重排已授权候选，返回 rank、raw_score、可选 calibrated_score、model_version、feature_version 和 trace。

离线样本以 query-doc pair 为事实单元，至少包含 query_group_id、event_time、doc_version、relevance_grade、judgment_source。缺失 judgment 是 UNKNOWN，不等于 0。训练、阈值选择与最终测试要使用不同 query group。

In [ ]:
from dataclasses import dataclass, asdict  # 导入本单元所需的依赖。
from collections import Counter, defaultdict  # 导入本单元所需的依赖。
from datetime import datetime  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import re  # 导入本单元所需的依赖。
import statistics  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

MODEL_VERSION = 'linear-pairwise-v1'  # 计算并保存当前步骤的中间状态。
FEATURE_VERSION = 'tfidf-features-v2'  # 计算并保存当前步骤的中间状态。
INDEX_VERSION = 'docs-2026-07'  # 计算并保存当前步骤的中间状态。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Query:  # 定义承载本节状态与行为的数据结构。
    query_id: str  # 执行当前语句以推进本节示例。
    group_id: str  # 执行当前语句以推进本节示例。
    event_time: str  # 执行当前语句以推进本节示例。
    tenant_id: str  # 执行当前语句以推进本节示例。
    text: str  # 执行当前语句以推进本节示例。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Document:  # 定义承载本节状态与行为的数据结构。
    doc_id: str  # 执行当前语句以推进本节示例。
    family_id: str  # 执行当前语句以推进本节示例。
    tenant_id: str  # 执行当前语句以推进本节示例。
    title: str  # 执行当前语句以推进本节示例。
    text: str  # 执行当前语句以推进本节示例。
    active: bool = True  # 计算并保存当前步骤的中间状态。

queries = [  # 计算并保存当前步骤的中间状态。
    Query('q-db-1', 'g-db', '2026-01-10', 'tenant-a', 'E1042 数据库连不上怎么办'),  # 执行当前语句以推进本节示例。
    Query('q-db-2', 'g-db', '2026-06-10', 'tenant-a', '数据库 E1042 连接超时'),  # 执行当前语句以推进本节示例。
    Query('q-cancel', 'g-cancel', '2026-02-05', 'tenant-a', '未发货订单怎么取消'),  # 执行当前语句以推进本节示例。
    Query('q-deploy', 'g-deploy', '2026-02-20', 'tenant-a', '发布失败如何回滚版本'),  # 执行当前语句以推进本节示例。
    Query('q-invoice', 'g-invoice', '2026-03-02', 'tenant-a', '电子发票在哪里下载'),  # 执行当前语句以推进本节示例。
    Query('q-refund', 'g-refund', '2026-04-12', 'tenant-a', '退款多久可以到账'),  # 执行当前语句以推进本节示例。
    Query('q-login', 'g-login', '2026-06-18', 'tenant-a', '忘记密码无法登录'),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
documents = [  # 计算并保存当前步骤的中间状态。
    Document('d-e1042', 'fam-e1042', 'tenant-a', 'E1042 数据库连接故障', '检查网络、连接串与连接池超时。'),  # 执行当前语句以推进本节示例。
    Document('d-e1042-copy', 'fam-e1042', 'tenant-a', '数据库故障 E1042', '连接失败时检查网络和连接池。'),  # 执行当前语句以推进本节示例。
    Document('d-slow-sql', 'fam-sql', 'tenant-a', 'SQL 慢查询', '检查索引、执行计划与锁等待。'),  # 执行当前语句以推进本节示例。
    Document('d-cancel', 'fam-cancel', 'tenant-a', '取消未发货订单', '订单详情页可取消，已发货需退货。'),  # 执行当前语句以推进本节示例。
    Document('d-deploy', 'fam-deploy', 'tenant-a', '发布回滚手册', '部署失败后选择上一稳定版本回滚。'),  # 执行当前语句以推进本节示例。
    Document('d-invoice', 'fam-invoice', 'tenant-a', '下载电子发票', '订单完成后在发票中心下载 PDF。'),  # 执行当前语句以推进本节示例。
    Document('d-refund', 'fam-refund', 'tenant-a', '退款到账时间', '审核后通常五个工作日原路到账。'),  # 执行当前语句以推进本节示例。
    Document('d-reset', 'fam-login', 'tenant-a', '忘记密码与登录失败', '使用短信验证码重置密码后重新登录。'),  # 执行当前语句以推进本节示例。
    Document('d-lock', 'fam-lock', 'tenant-a', '账号锁定', '连续输错密码会锁定账号，等待解锁。'),  # 执行当前语句以推进本节示例。
    Document('d-secret-b', 'fam-secret', 'tenant-b', '租户 B 密钥', '内部密钥 SECRET-B。'),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
by_qid = {q.query_id: q for q in queries}  # 计算并保存当前步骤的中间状态。
by_docid = {d.doc_id: d for d in documents}  # 计算并保存当前步骤的中间状态。
assert len(by_qid) == len(queries)  # 用受控断言验证关键不变量。
assert len(by_docid) == len(documents)  # 用受控断言验证关键不变量。
print('queries:', len(queries), 'documents:', len(documents))  # 执行当前语句以推进本节示例。

## 2. Pair schema 与 judgment 语义

相关性可用 0/1，也可用 0–3 分级。0 表示人工或可靠流程确认不相关；None 表示未判断。点击不是天然相关标签：位置偏差、曝光偏差、误点和满意后不点击都要处理。

示例只审计每个 query 的部分 pair，并故意让 d-e1042-copy 未标注，模拟潜在 false negative。训练采样器只能从明确 grade=0 的集合取负例。

In [ ]:
positive_grades = {  # 计算并保存当前步骤的中间状态。
    ('q-db-1', 'd-e1042'): 3, ('q-db-2', 'd-e1042'): 3,  # 执行当前语句以推进本节示例。
    ('q-cancel', 'd-cancel'): 3, ('q-deploy', 'd-deploy'): 3,  # 执行当前语句以推进本节示例。
    ('q-invoice', 'd-invoice'): 3, ('q-refund', 'd-refund'): 3,  # 执行当前语句以推进本节示例。
    ('q-login', 'd-reset'): 3, ('q-login', 'd-lock'): 1,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
judgments = dict(positive_grades)  # 计算并保存当前步骤的中间状态。
audited_zeroes = {  # 计算并保存当前步骤的中间状态。
    'q-db-1': ['d-slow-sql', 'd-cancel', 'd-deploy', 'd-invoice', 'd-refund', 'd-reset'],  # 执行当前语句以推进本节示例。
    'q-db-2': ['d-slow-sql', 'd-cancel', 'd-deploy', 'd-invoice', 'd-refund', 'd-reset'],  # 执行当前语句以推进本节示例。
    'q-cancel': ['d-slow-sql', 'd-deploy', 'd-invoice', 'd-refund', 'd-reset'],  # 执行当前语句以推进本节示例。
    'q-deploy': ['d-slow-sql', 'd-cancel', 'd-invoice', 'd-refund', 'd-reset'],  # 执行当前语句以推进本节示例。
    'q-invoice': ['d-slow-sql', 'd-cancel', 'd-deploy', 'd-refund', 'd-reset'],  # 执行当前语句以推进本节示例。
    'q-refund': ['d-slow-sql', 'd-cancel', 'd-deploy', 'd-invoice', 'd-reset'],  # 执行当前语句以推进本节示例。
    'q-login': ['d-slow-sql', 'd-cancel', 'd-deploy', 'd-invoice', 'd-refund'],  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
for qid, doc_ids in audited_zeroes.items():  # 遍历输入元素以累积或检查结果。
    for doc_id in doc_ids:  # 遍历输入元素以累积或检查结果。
        judgments[(qid, doc_id)] = 0  # 计算并保存当前步骤的中间状态。

assert judgments.get(('q-db-1', 'd-e1042-copy')) is None  # 用受控断言验证关键不变量。
assert judgments[('q-login', 'd-lock')] == 1  # 用受控断言验证关键不变量。
assert judgments[('q-refund', 'd-invoice')] == 0  # 用受控断言验证关键不变量。
print('judged pairs:', len(judgments), 'unknown pairs:',  # 执行当前语句以推进本节示例。
      len(queries)*len(documents)-len(judgments))  # 执行当前语句以推进本节示例。

## 3. Group + time split：同时满足族隔离与严格时序

相似 query 改写、同一会话、同一用户任务不能跨 split。若逐 query 按日期切，q-db-1 会进 train、q-db-2 会进 test，造成模板泄漏；但若按 group 最早时间把整个组放入 train，q-db-2 的未来事件又会泄漏到更早的 validation。

下面采用保守策略：只有整个 group 都落在同一时间窗才进入 train/valid/test；跨越 cutoff 的 group 标记为 `purged`，不参与训练、调参或最终指标。线上也可重定义更精确的 group，但不能通过未来数据倒灌来换取样本量。真实系统还要冻结文档、特征与 judgment 的 as-of snapshot。

In [ ]:
TRAIN_END = datetime(2026, 4, 1)  # 计算并保存当前步骤的中间状态。
VALID_END = datetime(2026, 6, 1)  # 计算并保存当前步骤的中间状态。

def group_time_split(queries):  # 定义本节可复用的核心函数。
    group_times = defaultdict(list)  # 计算并保存当前步骤的中间状态。
    for query in queries:  # 遍历输入元素以累积或检查结果。
        group_times[query.group_id].append(datetime.fromisoformat(query.event_time))  # 执行当前语句以推进本节示例。
    group_split = {}  # 计算并保存当前步骤的中间状态。
    for group_id, times in group_times.items():  # 遍历输入元素以累积或检查结果。
        earliest, latest = min(times), max(times)  # 计算并保存当前步骤的中间状态。
        if latest < TRAIN_END:  # 按当前条件选择后续控制路径。
            split = 'train'  # 计算并保存当前步骤的中间状态。
        elif earliest >= TRAIN_END and latest < VALID_END:  # 按当前条件选择后续控制路径。
            split = 'valid'  # 计算并保存当前步骤的中间状态。
        elif earliest >= VALID_END:  # 按当前条件选择后续控制路径。
            split = 'test'  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            split = 'purged'  # 计算并保存当前步骤的中间状态。
        group_split[group_id] = split  # 计算并保存当前步骤的中间状态。
    return {query.query_id: group_split[query.group_id] for query in queries}  # 返回当前分支计算出的结果。

query_split = group_time_split(queries)  # 计算并保存当前步骤的中间状态。
group_parts = defaultdict(set)  # 计算并保存当前步骤的中间状态。
for query in queries:  # 遍历输入元素以累积或检查结果。
    group_parts[query.group_id].add(query_split[query.query_id])  # 执行当前语句以推进本节示例。

active_times = {part: [datetime.fromisoformat(q.event_time) for q in queries  # 计算并保存当前步骤的中间状态。
                       if query_split[q.query_id] == part]  # 按当前条件选择后续控制路径。
                for part in ('train', 'valid', 'test')}  # 遍历输入元素以累积或检查结果。
assert query_split['q-db-1'] == query_split['q-db-2'] == 'purged'  # 用受控断言验证关键不变量。
assert query_split['q-refund'] == 'valid'  # 用受控断言验证关键不变量。
assert query_split['q-login'] == 'test'  # 用受控断言验证关键不变量。
assert all(len(parts) == 1 for parts in group_parts.values())  # 用受控断言验证关键不变量。
assert max(active_times['train']) < min(active_times['valid'])  # 用受控断言验证关键不变量。
assert max(active_times['valid']) < min(active_times['test'])  # 用受控断言验证关键不变量。
print(query_split)  # 执行当前语句以推进本节示例。

## 4. 手写 TF-IDF baseline：训练/服务共用 analyzer

baseline 的价值是建立可解释下限和 hard-negative 矿工。中文 analyzer 使用小领域词典最长匹配，ASCII 错误码整体保留；未知汉字退化为单字。生产必须对词典和归一化版本做回归。

IDF 使用可检索的 tenant-a active 文档集合。若语料库包含未来才发布的文档，时间评估需按当时 index snapshot 重建，不能让测试时点看到未来文档。

In [ ]:
LEXICON = sorted({  # 计算并保存当前步骤的中间状态。
    '数据库', '连接', '连接池', '超时', '检查', '网络', '慢查询', '索引', '执行计划',  # 执行当前语句以推进本节示例。
    '未发货', '订单', '取消', '退货', '发布', '部署', '失败', '回滚', '版本',  # 执行当前语句以推进本节示例。
    '电子发票', '发票中心', '下载', '退款', '到账', '工作日', '忘记密码',  # 执行当前语句以推进本节示例。
    '无法登录', '登录', '重置密码', '账号', '锁定', '验证码'  # 执行当前语句以推进本节示例。
}, key=len, reverse=True)  # 计算并保存当前步骤的中间状态。

def analyze(text):  # 定义本节可复用的核心函数。
    text = text.casefold()  # 计算并保存当前步骤的中间状态。
    tokens, i = [], 0  # 计算并保存当前步骤的中间状态。
    while i < len(text):  # 在终止条件满足前持续推进状态。
        if text[i].isspace() or text[i] in '，。：；！？、（）()/.':  # 按当前条件选择后续控制路径。
            i += 1  # 计算并保存当前步骤的中间状态。
            continue  # 调整当前循环或占位控制流。
        match = re.match(r'[a-z]+\d+|[a-z]+|\d+(?:\.\d+)?', text[i:])  # 计算并保存当前步骤的中间状态。
        if match:  # 按当前条件选择后续控制路径。
            tokens.append(match.group())  # 执行当前语句以推进本节示例。
            i += len(match.group())  # 计算并保存当前步骤的中间状态。
            continue  # 调整当前循环或占位控制流。
        word = next((w for w in LEXICON if text.startswith(w, i)), None)  # 计算并保存当前步骤的中间状态。
        tokens.append(word or text[i])  # 执行当前语句以推进本节示例。
        i += len(word) if word else 1  # 计算并保存当前步骤的中间状态。
    return tokens  # 返回当前分支计算出的结果。

index_docs = [d for d in documents if d.tenant_id == 'tenant-a' and d.active]  # 计算并保存当前步骤的中间状态。
doc_tokens = {d.doc_id: analyze(d.title + ' ' + d.text) for d in index_docs}  # 计算并保存当前步骤的中间状态。
df = Counter()  # 计算并保存当前步骤的中间状态。
for tokens in doc_tokens.values():  # 遍历输入元素以累积或检查结果。
    df.update(set(tokens))  # 执行当前语句以推进本节示例。
N = len(index_docs)  # 计算并保存当前步骤的中间状态。
idf = {term: math.log((N + 1)/(freq + 1)) + 1 for term, freq in df.items()}  # 计算并保存当前步骤的中间状态。

def tfidf_vector(tokens):  # 定义本节可复用的核心函数。
    counts = Counter(tokens)  # 计算并保存当前步骤的中间状态。
    raw = {term: (1 + math.log(freq))*idf.get(term, math.log(N+1)+1)  # 计算并保存当前步骤的中间状态。
           for term, freq in counts.items()}  # 遍历输入元素以累积或检查结果。
    norm = math.sqrt(sum(value*value for value in raw.values())) or 1.0  # 计算并保存当前步骤的中间状态。
    return {term: value/norm for term, value in raw.items()}  # 返回当前分支计算出的结果。

doc_vectors = {doc_id: tfidf_vector(tokens) for doc_id, tokens in doc_tokens.items()}  # 计算并保存当前步骤的中间状态。

def cosine_sparse(left, right):  # 定义本节可复用的核心函数。
    if len(left) > len(right):  # 按当前条件选择后续控制路径。
        left, right = right, left  # 计算并保存当前步骤的中间状态。
    return sum(value * right.get(term, 0.0) for term, value in left.items())  # 返回当前分支计算出的结果。

def lexical_score(query, document):  # 定义本节可复用的核心函数。
    return cosine_sparse(tfidf_vector(analyze(query)),  # 返回当前分支计算出的结果。
                         doc_vectors[document.doc_id])  # 执行当前语句以推进本节示例。

def baseline_rank(query, top_k=None):  # 定义本节可复用的核心函数。
    rows = [(d.doc_id, lexical_score(query.text, d))  # 计算并保存当前步骤的中间状态。
            for d in index_docs if d.tenant_id == query.tenant_id]  # 遍历输入元素以累积或检查结果。
    rows.sort(key=lambda row: (-row[1], row[0]))  # 计算并保存当前步骤的中间状态。
    return rows[:top_k] if top_k else rows  # 返回当前分支计算出的结果。

assert analyze('E1042 数据库连接')[:2] == ['e1042', '数据库']  # 用受控断言验证关键不变量。
assert baseline_rank(by_qid['q-refund'])[0][0] == 'd-refund'  # 用受控断言验证关键不变量。
assert all(doc_id != 'd-secret-b' for doc_id, _ in baseline_rank(by_qid['q-db-1']))  # 用受控断言验证关键不变量。
print(baseline_rank(by_qid['q-db-1'], 5))  # 执行当前语句以推进本节示例。

## 5. Pointwise 与 pairwise 的优化目标不同

pointwise 把每个 pair 当分类/回归样本，学习 P(relevant|q,d)；实现简单，但损失不直接关心同 query 内顺序，且负例数量会支配梯度。

pairwise 对同一 query 的正负文档差向量优化，目标是 score(q,d+) > score(q,d−)。它更贴近相对排序，却不会自动得到跨 query 可比较的概率。listwise 方法进一步直接处理整个列表或排序指标。

In [ ]:
def exact_identifiers(text):  # 定义本节可复用的核心函数。
    return set(re.findall(r'[a-z]+\d+|\d+(?:\.\d+)?', text.casefold()))  # 返回当前分支计算出的结果。

def feature_vector(query, document):  # 定义本节可复用的核心函数。
    q_tokens = set(analyze(query.text))  # 计算并保存当前步骤的中间状态。
    d_tokens = set(doc_tokens[document.doc_id])  # 计算并保存当前步骤的中间状态。
    coverage = len(q_tokens & d_tokens) / max(1, len(q_tokens))  # 计算并保存当前步骤的中间状态。
    exact_id = float(bool(exact_identifiers(query.text)) and  # 计算并保存当前步骤的中间状态。
                     exact_identifiers(query.text) <= exact_identifiers(document.title + ' ' + document.text))  # 计算并保存当前步骤的中间状态。
    title_overlap = len(q_tokens & set(analyze(document.title))) / max(1, len(q_tokens))  # 计算并保存当前步骤的中间状态。
    length_penalty = min(len(doc_tokens[document.doc_id]), 40) / 40  # 计算并保存当前步骤的中间状态。
    return np.array([  # 返回当前分支计算出的结果。
        lexical_score(query.text, document),  # 执行当前语句以推进本节示例。
        coverage,  # 执行当前语句以推进本节示例。
        exact_id,  # 执行当前语句以推进本节示例。
        title_overlap,  # 执行当前语句以推进本节示例。
        length_penalty,  # 执行当前语句以推进本节示例。
        1.0,  # 执行当前语句以推进本节示例。
    ], dtype=float)  # 计算并保存当前步骤的中间状态。

FEATURE_NAMES = ['tfidf_cosine', 'coverage', 'exact_id', 'title_overlap', 'length_scaled', 'bias']  # 计算并保存当前步骤的中间状态。
assert feature_vector(by_qid['q-db-1'], by_docid['d-e1042']).shape == (6,)  # 用受控断言验证关键不变量。
assert feature_vector(by_qid['q-db-1'], by_docid['d-e1042'])[2] == 1.0  # 用受控断言验证关键不变量。
print(dict(zip(FEATURE_NAMES, feature_vector(by_qid['q-db-1'], by_docid['d-e1042']))))  # 执行当前语句以推进本节示例。

## 6. Random 与 hard negatives

random negative 往往太容易，模型只学到主题词；hard negative 是 baseline 排名前列但经审计不相关的文档，能训练细粒度区分。越 hard 越可能是未标注正例，必须做 false-negative 防护：

- 只取明确 judgment=0，而非把 UNKNOWN 当 0。
- 排除正例同 document family、同 canonical URL/near duplicate。
- 对高分未知样本送人工复核或 teacher，而不是自动负标。
- mining index/model version 写入训练 manifest，防止不可回放。

为使 fixture 跨进程可重放，`random` 模式不是依赖全局 PRNG 状态，而是按 seed/query/doc 的 SHA-256 排序得到伪随机置换；回归同时检查固定期望与换 seed 后的变化。

In [ ]:
def positive_docs(qid):  # 定义本节可复用的核心函数。
    return [doc_id for (query_id, doc_id), grade in judgments.items()  # 返回当前分支计算出的结果。
            if query_id == qid and grade > 0]  # 按当前条件选择后续控制路径。

def eligible_negatives(qid):  # 定义本节可复用的核心函数。
    positive_families = {by_docid[doc_id].family_id for doc_id in positive_docs(qid)}  # 计算并保存当前步骤的中间状态。
    return [d for d in index_docs  # 返回当前分支计算出的结果。
            if judgments.get((qid, d.doc_id)) == 0  # 按当前条件选择后续控制路径。
            and d.family_id not in positive_families]  # 执行当前语句以推进本节示例。

def mine_negatives(qid, mode='hard', k=2, seed=7):  # 定义本节可复用的核心函数。
    query = by_qid[qid]  # 计算并保存当前步骤的中间状态。
    eligible = eligible_negatives(qid)  # 计算并保存当前步骤的中间状态。
    if mode == 'random':  # 按当前条件选择后续控制路径。
        ranked = sorted(eligible, key=lambda d: (  # 计算并保存当前步骤的中间状态。
            hashlib.sha256(f'{seed}:{qid}:{d.doc_id}'.encode()).hexdigest(), d.doc_id))  # 执行当前语句以推进本节示例。
        return [d.doc_id for d in ranked[:k]]  # 返回当前分支计算出的结果。
    if mode == 'hard':  # 按当前条件选择后续控制路径。
        ranked = sorted(eligible, key=lambda d: (-lexical_score(query.text, d), d.doc_id))  # 计算并保存当前步骤的中间状态。
        return [d.doc_id for d in ranked[:k]]  # 返回当前分支计算出的结果。
    raise ValueError('unknown_mode')  # 遇到非法合同立即显式失败。

hard_db = mine_negatives('q-db-1', 'hard', 3)  # 计算并保存当前步骤的中间状态。
random_db = mine_negatives('q-db-1', 'random', 3)  # 计算并保存当前步骤的中间状态。
assert 'd-e1042-copy' not in hard_db  # 用受控断言验证关键不变量。
assert 'd-e1042-copy' not in random_db  # 用受控断言验证关键不变量。
assert mine_negatives('q-db-1', 'random', 2, seed=7) == ['d-refund', 'd-deploy']  # 用受控断言验证关键不变量。
assert mine_negatives('q-db-1', 'random', 2, seed=8) == ['d-deploy', 'd-cancel']  # 用受控断言验证关键不变量。
assert all(judgments[('q-db-1', doc_id)] == 0 for doc_id in hard_db)  # 用受控断言验证关键不变量。
assert all(by_docid[doc_id].family_id != 'fam-e1042' for doc_id in hard_db)  # 用受控断言验证关键不变量。
print({'hard': hard_db, 'random': random_db})  # 执行当前语句以推进本节示例。

## 7. 从零实现 pointwise 与 pairwise 线性训练

下面用 NumPy 显式实现 logistic gradient descent，不依赖现成 ranker。特征量级较接近，但生产仍需在训练 artifact 中冻结归一化统计。

训练只使用 train query groups；valid 只用于校准和阈值；test 最后一次报告。为控制 class balance，每个正例配固定数量 hard negatives，并记录采样策略。

In [ ]:
train_qids = [q.query_id for q in queries if query_split[q.query_id] == 'train']  # 计算并保存当前步骤的中间状态。

def sigmoid(x):  # 定义本节可复用的核心函数。
    x = np.clip(x, -30, 30)  # 计算并保存当前步骤的中间状态。
    return 1 / (1 + np.exp(-x))  # 返回当前分支计算出的结果。

point_x, point_y, pair_x = [], [], []  # 计算并保存当前步骤的中间状态。
training_manifest = []  # 计算并保存当前步骤的中间状态。
for qid in train_qids:  # 遍历输入元素以累积或检查结果。
    query = by_qid[qid]  # 计算并保存当前步骤的中间状态。
    positives = positive_docs(qid)  # 计算并保存当前步骤的中间状态。
    negatives = mine_negatives(qid, 'hard', 3)  # 计算并保存当前步骤的中间状态。
    for pos_id in positives:  # 遍历输入元素以累积或检查结果。
        pos_features = feature_vector(query, by_docid[pos_id])  # 计算并保存当前步骤的中间状态。
        point_x.append(pos_features); point_y.append(1.0)  # 执行当前语句以推进本节示例。
        for neg_id in negatives:  # 遍历输入元素以累积或检查结果。
            neg_features = feature_vector(query, by_docid[neg_id])  # 计算并保存当前步骤的中间状态。
            point_x.append(neg_features); point_y.append(0.0)  # 执行当前语句以推进本节示例。
            pair_x.append(pos_features - neg_features)  # 执行当前语句以推进本节示例。
    training_manifest.append({'query_id': qid, 'positives': positives, 'negatives': negatives})  # 执行当前语句以推进本节示例。

point_x, point_y = np.vstack(point_x), np.array(point_y)  # 计算并保存当前步骤的中间状态。
pair_x = np.vstack(pair_x)  # 计算并保存当前步骤的中间状态。

def fit_pointwise(x, y, steps=1200, lr=0.12, l2=0.01):  # 定义本节可复用的核心函数。
    w = np.zeros(x.shape[1])  # 计算并保存当前步骤的中间状态。
    for _ in range(steps):  # 遍历输入元素以累积或检查结果。
        gradient = x.T @ (sigmoid(x @ w) - y) / len(y) + l2*w  # 计算并保存当前步骤的中间状态。
        w -= lr * gradient  # 计算并保存当前步骤的中间状态。
    return w  # 返回当前分支计算出的结果。

def fit_pairwise(differences, steps=1200, lr=0.12, l2=0.01):  # 定义本节可复用的核心函数。
    w = np.zeros(differences.shape[1])  # 计算并保存当前步骤的中间状态。
    for _ in range(steps):  # 遍历输入元素以累积或检查结果。
        # -log sigmoid(w·(x_pos-x_neg)) 中文说明：该注释解释本行约束。
        margins = differences @ w  # 计算并保存当前步骤的中间状态。
        gradient = -(differences.T @ (1 - sigmoid(margins))) / len(differences) + l2*w  # 计算并保存当前步骤的中间状态。
        w -= lr * gradient  # 计算并保存当前步骤的中间状态。
    return w  # 返回当前分支计算出的结果。

w_point = fit_pointwise(point_x, point_y)  # 计算并保存当前步骤的中间状态。
w_pair = fit_pairwise(pair_x)  # 计算并保存当前步骤的中间状态。
assert np.mean(pair_x @ w_pair > 0) >= 0.9  # 用受控断言验证关键不变量。
assert set(row['query_id'] for row in training_manifest) == set(train_qids)  # 用受控断言验证关键不变量。
print('pointwise:', dict(zip(FEATURE_NAMES, w_point.round(3))))  # 执行当前语句以推进本节示例。
print('pairwise:', dict(zip(FEATURE_NAMES, w_pair.round(3))))  # 执行当前语句以推进本节示例。

## 8. 排序函数：只重排候选，不绕过 ACL

reranker 的输入应是检索器已经按 tenant、active、时间/产品过滤后的 candidate IDs。服务端再次校验 scope，防御缓存键或调用方错误；不能让高分文档绕过硬权限。

pairwise raw score 只在同 query 候选间用于排序。确定性 tie-break 使用 doc_id，避免并行执行时排名抖动。

In [ ]:
def rerank(query, candidate_ids, weights=w_pair):  # 定义本节可复用的核心函数。
    rows = []  # 计算并保存当前步骤的中间状态。
    for doc_id in candidate_ids:  # 遍历输入元素以累积或检查结果。
        doc = by_docid[doc_id]  # 计算并保存当前步骤的中间状态。
        if doc.tenant_id != query.tenant_id or not doc.active:  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        features = feature_vector(query, doc)  # 计算并保存当前步骤的中间状态。
        rows.append({  # 执行当前语句以推进本节示例。
            'doc_id': doc_id,  # 执行当前语句以推进本节示例。
            'raw_score': float(features @ weights),  # 执行当前语句以推进本节示例。
            'features': dict(zip(FEATURE_NAMES, features.tolist())),  # 执行当前语句以推进本节示例。
        })  # 执行当前语句以推进本节示例。
    return sorted(rows, key=lambda row: (-row['raw_score'], row['doc_id']))  # 返回当前分支计算出的结果。

candidate_ids = [doc_id for doc_id, _ in baseline_rank(by_qid['q-refund'], 6)] + ['d-secret-b']  # 计算并保存当前步骤的中间状态。
ranked_refund = rerank(by_qid['q-refund'], candidate_ids)  # 计算并保存当前步骤的中间状态。
assert all(row['doc_id'] != 'd-secret-b' for row in ranked_refund)  # 用受控断言验证关键不变量。
assert ranked_refund[0]['doc_id'] == 'd-refund'  # 用受控断言验证关键不变量。
print(ranked_refund[:3])  # 执行当前语句以推进本节示例。

## 9. Recall、MRR 与 nDCG：UNKNOWN 不能偷偷变成 0

Recall@k 判断相关文档是否进入前 k；MRR 只关心第一个相关结果；nDCG 使用分级 gain 和位置折扣。评估单元必须是 query group，不能把每个 pair 当独立样本算 accuracy。

本 fixture 的 judgment 不完整，因此采用 judged-only 协议：先从排名中跳过 UNKNOWN，再计算指标；同时报告原始 top-k 的 judgment coverage。此时指标中的 k 是“已判断位置”，不能与把 UNKNOWN 当 0 的官方全量评测直接横比。coverage 太低时应扩充 pooling/人工标注，或采用 bpref 等不完整判断指标，不能只展示漂亮的 MRR。

In [ ]:
def judged_ranking(ranking, grades):  # 定义本节可复用的核心函数。
    return [doc_id for doc_id in ranking if doc_id in grades]  # 返回当前分支计算出的结果。

def judgment_coverage_at_k(ranking, grades, k):  # 定义本节可复用的核心函数。
    head = ranking[:k]  # 计算并保存当前步骤的中间状态。
    return sum(doc_id in grades for doc_id in head) / len(head) if head else None  # 返回当前分支计算出的结果。

def recall_at_k(ranking, grades, k):  # 定义本节可复用的核心函数。
    relevant = {doc_id for doc_id, grade in grades.items() if grade > 0}  # 计算并保存当前步骤的中间状态。
    if not relevant:  # 按当前条件选择后续控制路径。
        return None  # 返回当前分支计算出的结果。
    evaluated = judged_ranking(ranking, grades)  # 计算并保存当前步骤的中间状态。
    return len(set(evaluated[:k]) & relevant) / len(relevant)  # 返回当前分支计算出的结果。

def reciprocal_rank(ranking, grades):  # 定义本节可复用的核心函数。
    for rank, doc_id in enumerate(judged_ranking(ranking, grades), 1):  # 遍历输入元素以累积或检查结果。
        if grades[doc_id] > 0:  # 按当前条件选择后续控制路径。
            return 1 / rank  # 返回当前分支计算出的结果。
    return 0.0  # 返回当前分支计算出的结果。

def ndcg_at_k(ranking, grades, k):  # 定义本节可复用的核心函数。
    def dcg(items):  # 定义本节可复用的核心函数。
        return sum((2**grades[doc_id]-1) / math.log2(rank+1)  # 返回当前分支计算出的结果。
                   for rank, doc_id in enumerate(items, 1))  # 遍历输入元素以累积或检查结果。
    evaluated = judged_ranking(ranking, grades)[:k]  # 计算并保存当前步骤的中间状态。
    ideal = sorted(grades, key=lambda d: (-grades[d], d))[:k]  # 计算并保存当前步骤的中间状态。
    denominator = dcg(ideal)  # 计算并保存当前步骤的中间状态。
    return dcg(evaluated) / denominator if denominator else None  # 返回当前分支计算出的结果。

def evaluate_query(qid, candidate_n=8):  # 定义本节可复用的核心函数。
    query = by_qid[qid]  # 计算并保存当前步骤的中间状态。
    candidates = [doc_id for doc_id, _ in baseline_rank(query, candidate_n)]  # 计算并保存当前步骤的中间状态。
    ranking = [row['doc_id'] for row in rerank(query, candidates)]  # 计算并保存当前步骤的中间状态。
    grades = {doc_id: grade for (query_id, doc_id), grade in judgments.items()  # 计算并保存当前步骤的中间状态。
              if query_id == qid}  # 按当前条件选择后续控制路径。
    evaluated = judged_ranking(ranking, grades)  # 计算并保存当前步骤的中间状态。
    return {'query_id': qid, 'raw_ranking': ranking, 'judged_ranking': evaluated,  # 返回当前分支计算出的结果。
            'judgment_coverage@5': judgment_coverage_at_k(ranking, grades, 5),  # 执行当前语句以推进本节示例。
            'recall@3': recall_at_k(ranking, grades, 3),  # 执行当前语句以推进本节示例。
            'mrr': reciprocal_rank(ranking, grades),  # 执行当前语句以推进本节示例。
            'ndcg@5': ndcg_at_k(ranking, grades, 5)}  # 执行当前语句以推进本节示例。

valid_report = evaluate_query('q-refund')  # 计算并保存当前步骤的中间状态。
test_report = evaluate_query('q-login')  # 计算并保存当前步骤的中间状态。
assert valid_report['recall@3'] == 1.0  # 用受控断言验证关键不变量。
assert 0 < valid_report['judgment_coverage@5'] <= 1  # 用受控断言验证关键不变量。
assert 0 <= test_report['ndcg@5'] <= 1  # 用受控断言验证关键不变量。
assert 0 <= test_report['mrr'] <= 1  # 用受控断言验证关键不变量。
print({'valid': valid_report, 'test': test_report})  # 执行当前语句以推进本节示例。

## 10. 校准与阈值：不要把 raw rank score 当概率

排序正确不代表分数可跨 query 比较。若业务要“低置信拒绝/不调用昂贵下游”，需要在独立 validation 集上校准。这里用 Platt-style sigmoid a·score+b 的手写优化演示接口。

一个 validation query 太少，结果只用于代码路径；生产要按 query 类型/语言/流量桶使用足量数据，阈值与 model/index/calibrator 版本绑定。测试集不能参与阈值挑选。

In [ ]:
def validation_pairs(qids):  # 定义本节可复用的核心函数。
    scores, labels = [], []  # 计算并保存当前步骤的中间状态。
    for qid in qids:  # 遍历输入元素以累积或检查结果。
        query = by_qid[qid]  # 计算并保存当前步骤的中间状态。
        for doc in index_docs:  # 遍历输入元素以累积或检查结果。
            grade = judgments.get((qid, doc.doc_id))  # 计算并保存当前步骤的中间状态。
            if grade is None:  # 按当前条件选择后续控制路径。
                continue  # 调整当前循环或占位控制流。
            scores.append(float(feature_vector(query, doc) @ w_pair))  # 执行当前语句以推进本节示例。
            labels.append(float(grade > 0))  # 执行当前语句以推进本节示例。
    return np.array(scores), np.array(labels)  # 返回当前分支计算出的结果。

def fit_platt(scores, labels, steps=1500, lr=0.08):  # 定义本节可复用的核心函数。
    a, b = 0.0, math.log((labels.mean()+0.05)/(1-labels.mean()+0.05))  # 计算并保存当前步骤的中间状态。
    for _ in range(steps):  # 遍历输入元素以累积或检查结果。
        probs = sigmoid(a*scores + b)  # 计算并保存当前步骤的中间状态。
        a -= lr * (np.mean((probs-labels)*scores) + 0.001*a)  # 计算并保存当前步骤的中间状态。
        b -= lr * np.mean(probs-labels)  # 计算并保存当前步骤的中间状态。
    return float(a), float(b)  # 返回当前分支计算出的结果。

valid_scores, valid_labels = validation_pairs(['q-refund'])  # 计算并保存当前步骤的中间状态。
cal_a, cal_b = fit_platt(valid_scores, valid_labels)  # 计算并保存当前步骤的中间状态。
valid_probs = sigmoid(cal_a*valid_scores + cal_b)  # 计算并保存当前步骤的中间状态。
threshold = float(np.quantile(valid_probs[valid_labels == 1], 0.1))  # 计算并保存当前步骤的中间状态。
assert 0 < threshold < 1  # 用受控断言验证关键不变量。
assert not {'q-login'} & {'q-refund'}  # 用受控断言验证关键不变量。
print({'calibrator': (round(cal_a, 3), round(cal_b, 3)),  # 执行当前语句以推进本节示例。
       'threshold': round(threshold, 3), 'valid_probs': valid_probs.round(3).tolist()})  # 执行当前语句以推进本节示例。

## 11. Rerank budget 与 batch serving

cross-encoder 成本大致随候选数和序列长度增长。先根据 Recall@N 选择 candidate_n，再给 reranker 限定每批 max_pairs/max_tokens、请求级 max_total_pairs 与 deadline；不能为了省延迟悄悄截断而不打 degraded 标记。

动态 batch 应按 token budget 拼批。单个 pair 已超过预算时，本教学实现显式丢入 `dropped_oversize` 并降级；生产也可按可审计的截断策略处理，但绝不能让超预算样本悄悄穿透。请求级公平性同样重要。

In [ ]:
def estimate_tokens(text):  # 定义本节可复用的核心函数。
    return max(1, len(analyze(text)))  # 返回当前分支计算出的结果。

def pair_token_cost(query, doc_id):  # 定义本节可复用的核心函数。
    doc = by_docid[doc_id]  # 计算并保存当前步骤的中间状态。
    return estimate_tokens(query.text) + estimate_tokens(doc.title + doc.text)  # 返回当前分支计算出的结果。

def make_batches(query, candidate_ids, max_pairs=3, max_tokens=45, max_total_pairs=9):  # 定义本节可复用的核心函数。
    if min(max_pairs, max_tokens, max_total_pairs) <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError('budgets_must_be_positive')  # 遇到非法合同立即显式失败。
    batches, current, used, dropped_oversize = [], [], 0, []  # 计算并保存当前步骤的中间状态。
    selected = candidate_ids[:max_total_pairs]  # 计算并保存当前步骤的中间状态。
    for doc_id in selected:  # 遍历输入元素以累积或检查结果。
        cost = pair_token_cost(query, doc_id)  # 计算并保存当前步骤的中间状态。
        if cost > max_tokens:  # 按当前条件选择后续控制路径。
            dropped_oversize.append({'doc_id': doc_id, 'estimated_tokens': cost})  # 执行当前语句以推进本节示例。
            continue  # 调整当前循环或占位控制流。
        if current and (len(current) >= max_pairs or used + cost > max_tokens):  # 按当前条件选择后续控制路径。
            batches.append(current)  # 执行当前语句以推进本节示例。
            current, used = [], 0  # 计算并保存当前步骤的中间状态。
        current.append(doc_id)  # 执行当前语句以推进本节示例。
        used += cost  # 计算并保存当前步骤的中间状态。
    if current:  # 按当前条件选择后续控制路径。
        batches.append(current)  # 执行当前语句以推进本节示例。
    trace = {  # 计算并保存当前步骤的中间状态。
        'truncated': len(candidate_ids) > max_total_pairs,  # 执行当前语句以推进本节示例。
        'dropped_oversize': dropped_oversize,  # 执行当前语句以推进本节示例。
        'degraded': len(candidate_ids) > max_total_pairs or bool(dropped_oversize),  # 执行当前语句以推进本节示例。
        'max_pairs': max_pairs, 'max_tokens': max_tokens,  # 执行当前语句以推进本节示例。
        'max_total_pairs': max_total_pairs,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    return batches, trace  # 返回当前分支计算出的结果。

login_candidates = [doc_id for doc_id, _ in baseline_rank(by_qid['q-login'])]  # 计算并保存当前步骤的中间状态。
batches, budget_trace = make_batches(by_qid['q-login'], login_candidates)  # 计算并保存当前步骤的中间状态。
oversize_batches, oversize_trace = make_batches(  # 计算并保存当前步骤的中间状态。
    by_qid['q-login'], ['d-e1042'], max_pairs=3, max_tokens=1, max_total_pairs=3)  # 计算并保存当前步骤的中间状态。
assert all(len(batch) <= 3 for batch in batches)  # 用受控断言验证关键不变量。
assert all(sum(pair_token_cost(by_qid['q-login'], doc_id) for doc_id in batch) <= 45  # 用受控断言验证关键不变量。
           for batch in batches)  # 遍历输入元素以累积或检查结果。
assert sorted(doc for batch in batches for doc in batch) == sorted(login_candidates[:9])  # 用受控断言验证关键不变量。
assert oversize_batches == []  # 用受控断言验证关键不变量。
assert oversize_trace['degraded'] and oversize_trace['dropped_oversize'][0]['doc_id'] == 'd-e1042'  # 用受控断言验证关键不变量。
print(batches, budget_trace, oversize_trace)  # 执行当前语句以推进本节示例。

## 12. Trace、漂移与降级

线上 trace 至少记录：tenant/request、query bucket、retriever/index/model/calibrator 版本、候选数、截断原因、各阶段耗时、top score/margin、拒识原因。每次调用的 request_id 用 nonce 区分；可重试去重的 idempotency_key 则由 tenant/query/candidate/version 合同稳定计算，两者不能混为一个字段。正文和 PII 默认不落日志。

漂移同时看 feature/score 分布、hard-negative 率、候选 Recall 的人工抽检、点击位置偏差和 latency。下面用 PSI 演示分数分布变化；PSI 只是报警信号，不能证明相关性下降。

In [ ]:
def psi(expected, actual, bins=(-math.inf, -0.5, 0.0, 0.5, 1.0, math.inf), eps=1e-6):  # 定义本节可复用的核心函数。
    def proportions(values):  # 定义本节可复用的核心函数。
        counts = [0] * (len(bins)-1)  # 计算并保存当前步骤的中间状态。
        for value in values:  # 遍历输入元素以累积或检查结果。
            for i in range(len(counts)):  # 遍历输入元素以累积或检查结果。
                if bins[i] <= value < bins[i+1]:  # 按当前条件选择后续控制路径。
                    counts[i] += 1  # 计算并保存当前步骤的中间状态。
                    break  # 调整当前循环或占位控制流。
        total = max(1, len(values))  # 计算并保存当前步骤的中间状态。
        return [(count+eps)/(total+eps*len(counts)) for count in counts]  # 返回当前分支计算出的结果。
    p, q = proportions(expected), proportions(actual)  # 计算并保存当前步骤的中间状态。
    return sum((b-a)*math.log(b/a) for a, b in zip(p, q))  # 返回当前分支计算出的结果。

baseline_scores = [-0.8, -0.6, -0.2, 0.1, 0.4, 0.8, 1.0]  # 计算并保存当前步骤的中间状态。
stable_scores = [-0.7, -0.5, -0.1, 0.2, 0.5, 0.7, 0.9]  # 计算并保存当前步骤的中间状态。
shifted_scores = [0.7, 0.8, 1.0, 1.1, 1.2, 1.3, 1.4]  # 计算并保存当前步骤的中间状态。
assert psi(baseline_scores, shifted_scores) > psi(baseline_scores, stable_scores)  # 用受控断言验证关键不变量。

def idempotency_key(query, candidates):  # 定义本节可复用的核心函数。
    payload = json.dumps({  # 计算并保存当前步骤的中间状态。
        'tenant_id': query.tenant_id, 'query_id': query.query_id,  # 执行当前语句以推进本节示例。
        'candidate_ids': candidates, 'index_version': INDEX_VERSION,  # 执行当前语句以推进本节示例。
        'model_version': MODEL_VERSION, 'feature_version': FEATURE_VERSION,  # 执行当前语句以推进本节示例。
    }, ensure_ascii=False, sort_keys=True, separators=(',', ':'))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(payload.encode()).hexdigest()[:24]  # 返回当前分支计算出的结果。

def build_trace(query, candidates, ranking, request_nonce, degraded=False, reason=None):  # 定义本节可复用的核心函数。
    return {  # 返回当前分支计算出的结果。
        'request_id': hashlib.sha256(  # 执行当前语句以推进本节示例。
            f'{request_nonce}:{query.tenant_id}:{query.query_id}'.encode()).hexdigest()[:16],  # 执行当前语句以推进本节示例。
        'idempotency_key': idempotency_key(query, candidates),  # 执行当前语句以推进本节示例。
        'query_id': query.query_id, 'tenant_id': query.tenant_id,  # 执行当前语句以推进本节示例。
        'candidate_count': len(candidates), 'returned_count': len(ranking),  # 执行当前语句以推进本节示例。
        'index_version': INDEX_VERSION, 'model_version': MODEL_VERSION,  # 执行当前语句以推进本节示例。
        'feature_version': FEATURE_VERSION,  # 执行当前语句以推进本节示例。
        'top_margin': round(ranking[0]['raw_score']-ranking[1]['raw_score'], 4)  # 执行当前语句以推进本节示例。
                      if len(ranking) > 1 else None,  # 按当前条件选择后续控制路径。
        'degraded': degraded, 'degraded_reason': reason,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。

trace = build_trace(by_qid['q-refund'], candidate_ids, ranked_refund, 'fixture-request-001')  # 计算并保存当前步骤的中间状态。
retry_trace = build_trace(by_qid['q-refund'], candidate_ids, ranked_refund, 'fixture-request-002')  # 计算并保存当前步骤的中间状态。
assert trace['tenant_id'] == 'tenant-a' and 'text' not in trace  # 用受控断言验证关键不变量。
assert trace['request_id'] != retry_trace['request_id']  # 用受控断言验证关键不变量。
assert trace['idempotency_key'] == retry_trace['idempotency_key']  # 用受控断言验证关键不变量。
print(json.dumps(trace, ensure_ascii=False, indent=2))  # 计算并保存当前步骤的中间状态。

## 13. 失败反例清单

- 把所有未点击/未标注文档当负例：潜在正例被当作 hard negative。
- 逐 pair 随机切分：同一 query 或改写同时进入 train/test。
- 按 group 最早时间切分：同组未来事件倒灌到更早的 valid/test；跨 cutoff 组应 purge。
- 指标用 `grades.get(doc, 0)`：UNKNOWN 被偷偷当作不相关，必须配 judged-only/bpref 与 coverage。
- hard negative 用测试集线上排名挖掘：评估标签泄漏到训练。
- 在全库 rerank 后再 ACL：无权文档已进入模型、GPU batch 和日志。
- 直接相加 BM25、embedding、ranker raw score：量纲与版本漂移不可控。
- 用 test 调 candidate_n、threshold 或 early stopping：最终指标失真。
- 只报 pair accuracy：大量简单负例掩盖 top rank 失败。
- score 漂移就自动重训：可能只是候选分布、流量或 index 版本改变。

In [ ]:
# 跨模块可执行回归
assert query_split['q-db-1'] == query_split['q-db-2'] == 'purged'  # 用受控断言验证关键不变量。
assert set(train_qids).isdisjoint({'q-db-1', 'q-db-2', 'q-refund', 'q-login'})  # 用受控断言验证关键不变量。
assert max(active_times['train']) < min(active_times['valid']) < min(active_times['test'])  # 用受控断言验证关键不变量。
assert judgments.get(('q-db-1', 'd-e1042-copy')) is None  # 用受控断言验证关键不变量。
assert 'd-e1042-copy' not in eligible_negatives('q-db-1')  # 用受控断言验证关键不变量。
assert mine_negatives('q-db-1', 'random', 2, seed=7) == ['d-refund', 'd-deploy']  # 用受控断言验证关键不变量。
assert mine_negatives('q-db-1', 'random', 2, seed=8) != mine_negatives('q-db-1', 'random', 2, seed=7)  # 用受控断言验证关键不变量。
assert all(judgments[('q-cancel', d)] == 0 for d in mine_negatives('q-cancel', 'hard', 2))  # 用受控断言验证关键不变量。
assert feature_vector(by_qid['q-db-1'], by_docid['d-e1042'])[2] == 1  # 用受控断言验证关键不变量。
assert float(np.mean(pair_x @ w_pair > 0)) >= 0.9  # 用受控断言验证关键不变量。
assert rerank(by_qid['q-login'], ['d-secret-b']) == []  # 用受控断言验证关键不变量。
assert evaluate_query('q-refund')['recall@3'] == 1.0  # 用受控断言验证关键不变量。
assert recall_at_k(['a'], {}, 1) is None  # 用受控断言验证关键不变量。
assert reciprocal_rank(['unknown', 'relevant'], {'relevant': 1}) == 1.0  # 用受控断言验证关键不变量。
assert reciprocal_rank(['judged-zero', 'relevant'], {'judged-zero': 0, 'relevant': 1}) == 0.5  # 用受控断言验证关键不变量。
assert judgment_coverage_at_k(['unknown', 'relevant'], {'relevant': 1}, 2) == 0.5  # 用受控断言验证关键不变量。
assert ndcg_at_k(['a'], {'a': 3}, 1) == 1.0  # 用受控断言验证关键不变量。
assert 0 < threshold < 1  # 用受控断言验证关键不变量。
assert all(len(batch) <= 3 for batch in batches)  # 用受控断言验证关键不变量。
assert all(sum(pair_token_cost(by_qid['q-login'], doc_id) for doc_id in batch) <= 45  # 用受控断言验证关键不变量。
           for batch in batches)  # 遍历输入元素以累积或检查结果。
assert oversize_batches == [] and oversize_trace['degraded']  # 用受控断言验证关键不变量。
assert trace['model_version'] == MODEL_VERSION  # 用受控断言验证关键不变量。
assert trace['request_id'] != retry_trace['request_id']  # 用受控断言验证关键不变量。
assert trace['idempotency_key'] == retry_trace['idempotency_key']  # 用受控断言验证关键不变量。
print('文本排序工程合同回归：全部通过')  # 执行当前语句以推进本节示例。

## 14. 服务、版本、安全与回滚

模型 artifact 应冻结 feature schema/order、analyzer/词典、训练 manifest、negative miner、代码 commit、随机种子、权重与 calibrator；发布配置再绑定 index snapshot、candidate_n、rerank_n、阈值。

灰度时比较候选交集、交换对、MRR/nDCG 分桶、拒绝覆盖率、P95/P99 和业务护栏。回滚要同时回滚 model + feature + calibrator 配套版本。缓存键包含 tenant、query hash、candidate hash 和所有版本。训练日志不写 query/doc 正文；敏感 tenant 可独立模型/索引。

## 15. 现代模型替换点

- 手写 TF-IDF → BM25/SPLADE/dense/hybrid retriever，但候选 Recall 与 snapshot 合同保持。
- 线性 pairwise → cross-encoder（query 与 doc 联合编码）、late interaction 或 LLM reranker。
- 规则 hard negative → 版本化 retriever mining、teacher 筛选、去假负及定期人工审计。
- 单机逐 pair → length-aware dynamic batching、ONNX/TensorRT、量化、缓存与 deadline-aware degradation。
- Platt 小样本示范 → 独立且足量验证集上的 temperature/isotonic/Platt 校准。

cross-encoder 更贵，通常只处理 top-N；模型质量、candidate depth、序列截断和 batch latency 必须联合消融。

## 16. 原论文与官方资料

1. Robertson & Zaragoza, The Probabilistic Relevance Framework: BM25 and Beyond：https://doi.org/10.1561/1500000019
2. Burges et al., Learning to Rank using Gradient Descent（RankNet）：https://www.microsoft.com/en-us/research/publication/learning-to-rank-using-gradient-descent/
3. Järvelin & Kekäläinen, Cumulated Gain-Based Evaluation of IR Techniques（DCG/nDCG）：https://doi.org/10.1145/582415.582418
4. Nogueira & Cho, Passage Re-ranking with BERT：https://arxiv.org/abs/1901.04085
5. Xiong et al., Approximate Nearest Neighbor Negative Contrastive Learning for Dense Text Retrieval（ANCE）：https://arxiv.org/abs/2007.00808
6. scikit-learn calibration user guide（概念参考，本文为手写实现）：https://scikit-learn.org/stable/modules/calibration.html

边界：受控 fixture、线性特征和单 query 校准只验证实现；不能外推真实搜索收益，也不能取代真实标注、线上互换实验与安全审查。